We are interested in estimating the probability that some finite random walk $S_n$ exceeds a threshold $x$, that is, the maximum value of the random walk $S_n$ exceeds $x$.

$$P(max_n(S_n) > x)$$
This event can be understoods as; if the sequence at some point exceeds $x$, then the maximum of the sequence will exceed $x$. This is equivalent to the event that the random walk $S_n$ exceeds $x$ at some point in time, thus
$$P(max_n(S_n) > x) = P(\tau(x) < \infty)$$,
we can read is as:
$$
\sup_{n\geq1}S_n>x
\iff
\exists n\geq1:S_n>x
\iff
\tau(x)<\infty
$$
Since for finite $n$, we have that the supremum and the maximum are equivalent.

For $X_i \sim N(-1,1)$, we get the cumulant generating function
$$
\kappa(\theta) = \log E[e^{\theta X_i}] = -\theta + \frac{\theta^2}{2}
$$
This has a unique positive root $\theta^* = 2$ such that $\kappa(\theta^*) = 0$.

For a $N(-1,1)$ a tilt of $e^{\theta^* x}$ gives us a new distribution $N(1,1)$, which is the distribution we will sample from to estimate the probability of exceeding $x$.

Note for task 31, we set $\theta^* = 1$ and the new distribution is $N(0,1)$ and we get a new cumulant generating function $\kappa(\theta) = -\frac{\theta^2}{2}$. We observe that the number of steps is much larger for $\theta^* = 1$ than for $\theta^* = 2$, and for $x=10$ we run out of time to compute the estimate for $\theta^* = 1$.

In [22]:
import numpy as np
from PIL.ImImagePlugin import number


N = 100
theta_star = 2

def siegmund_algorithm(N ,theta_star, x, distribution, cumulant):
    """
    Siegmund's algorithm to estimate P(max_n(S_n) > x)
    Follows Handbook of Monte Carlo methods algorithm 10.1

    Input:
    N: number of simulations
    theta_star: root of cumlant generationg function of
    X_i where Sn = S(X_1, ..., X_n) = sum(X_i) from 1 to n
    x: threshold to exceed
    distribution: lambda function of to sample TILTED distribution
    (example lambda: np.random.normal(-1,1)

    """
    # Step 1 is to compute theta_star, done outside function.
    number_of_steps = []
    Zs = []
    for _ in range(N):
        # Step 2 perform random walk
        S = 0
        # Step 2
        n = 0
        while S < x:

            S += distribution()
            n += 1
        S_tau = S

        def W(S_tau,theta_star):
            """Input first Sn where Sn > x"""
            return np.exp(-theta_star*S_tau + n*cumulant(theta_star))

        Z = W(S_tau,theta_star)

        Zs.append(Z)
        number_of_steps.append(n)

    avg_number_of_steps = np.mean(number_of_steps)
    estimator = np.mean(Zs)
    return estimator, avg_number_of_steps

for theta_star in [2, 1]:
    if theta_star == 2:
        dist = lambda : np.random.normal(1,1)
    if theta_star == 1:
        dist = lambda : np.random.normal(0,1)

    print("For theta_star =", theta_star)
    print()
    for x in [1,10]:
        est, avg_nsteps = siegmund_algorithm(N, theta_star, x, dist, cumulant= lambda theta : -theta + theta_star**2 / 2 )
        print(f"Estimated probability P(max_n(S_n) > {x}) = {est}")
        print(f"Average number of steps to exceed {x} = {avg_nsteps}")

For theta_star = 2

Estimated probability P(max_n(S_n) > 1) = 0.04577147756934886
Average number of steps to exceed 1 = 1.7
Estimated probability P(max_n(S_n) > 10) = 5.796072387001271e-10
Average number of steps to exceed 10 = 10.92
For theta_star = 1

Estimated probability P(max_n(S_n) > 1) = 0.046509833513182226
Average number of steps to exceed 1 = 10658.42
Estimated probability P(max_n(S_n) > 10) = 3.812032438162831e-11
Average number of steps to exceed 10 = 11096.16


In [16]:
X = np.random.normal(-1,1, size  = (100, N))
# Cumlative sum across rows
S = np.cumsum(X, axis = 0)

Smax = np.max(S, axis = 0)



np.mean(Smax > 1), np.mean(Smax > 10)

(np.float64(0.042), np.float64(0.0))